# GenStudio prototype — validate all 3 models FREE on Colab

Runs the **exact same** ComfyUI + workflows the Cloud Run worker uses, on a free Colab T4.
Proves FLUX-schnell (image), LTX-Video (video), Kokoro (voice) all generate before you spend a cent.

**Runtime → Change runtime type → T4 GPU** first. Then Run all.

All 3 models are Apache-2.0 → output is legally yours to sell. Self-contained: no repo clone needed.

In [ ]:
# 0. Confirm GPU present
!nvidia-smi -L || print('NO GPU — set Runtime type to T4 GPU')

In [ ]:
# 1. Install ComfyUI + the two custom nodes the worker uses
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip -q install -r /content/ComfyUI/requirements.txt
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
    !pip -q install -r /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
    !git clone https://github.com/stavsap/comfyui-kokoro.git /content/ComfyUI/custom_nodes/comfyui-kokoro
    !pip -q install -r /content/ComfyUI/custom_nodes/comfyui-kokoro/requirements.txt
!pip -q install websocket-client

In [ ]:
# 2. Download the 3 Apache-2.0 models (fp8 where possible to fit T4 16GB)
C='/content/ComfyUI/models'
for d in ['checkpoints','text_encoders','kokoro']:
    os.makedirs(f'{C}/{d}', exist_ok=True)

def dl(url, path):
    if not os.path.exists(path):
        !wget -q --show-progress -O {path} "{url}"
    else:
        print('have', path)

dl('https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors',
   f'{C}/checkpoints/flux1-schnell-fp8.safetensors')
dl('https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors',
   f'{C}/checkpoints/ltx-video-2b-v0.9.5.safetensors')
dl('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
   f'{C}/text_encoders/t5xxl_fp8_e4m3fn.safetensors')
print('models ready')

In [ ]:
# 3. Start ComfyUI in the background, wait for its HTTP API
import subprocess, time, urllib.request
HOST='127.0.0.1:8188'
proc = subprocess.Popen(['python','/content/ComfyUI/main.py','--listen','127.0.0.1','--port','8188'],
                        cwd='/content/ComfyUI')
for _ in range(180):
    try:
        urllib.request.urlopen(f'http://{HOST}/system_stats', timeout=2); print('ComfyUI up'); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ComfyUI did not start — check logs above')

In [ ]:
# 4. Workflow templates + run() — mirrors worker/server.py.
#    Numeric tokens ($WIDTH/$FRAMES) sit BARE (unquoted) so we text-replace BEFORE json parse.
import json, uuid, base64, urllib.parse, websocket, os

TEMPLATES = {
"image": '''{
 "1": {"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":"flux1-schnell-fp8.safetensors"}},
 "2": {"class_type":"CLIPTextEncode","inputs":{"text":"$PROMPT","clip":["1",1]}},
 "3": {"class_type":"CLIPTextEncode","inputs":{"text":"","clip":["1",1]}},
 "4": {"class_type":"EmptyLatentImage","inputs":{"width":$WIDTH,"height":$HEIGHT,"batch_size":1}},
 "5": {"class_type":"KSampler","inputs":{"seed":$SEED,"steps":4,"cfg":1.0,"sampler_name":"euler","scheduler":"simple","denoise":1.0,"model":["1",0],"positive":["2",0],"negative":["3",0],"latent_image":["4",0]}},
 "6": {"class_type":"VAEDecode","inputs":{"samples":["5",0],"vae":["1",2]}},
 "7": {"class_type":"SaveImage","inputs":{"filename_prefix":"gen","images":["6",0]}}
}''',
"video": '''{
 "1": {"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":"ltx-video-2b-v0.9.5.safetensors"}},
 "2": {"class_type":"CLIPLoader","inputs":{"clip_name":"t5xxl_fp8_e4m3fn.safetensors","type":"ltxv"}},
 "3": {"class_type":"CLIPTextEncode","inputs":{"text":"$PROMPT","clip":["2",0]}},
 "4": {"class_type":"CLIPTextEncode","inputs":{"text":"low quality, worst quality, blurry","clip":["2",0]}},
 "5": {"class_type":"EmptyLTXVLatentVideo","inputs":{"width":$WIDTH,"height":$HEIGHT,"length":$FRAMES,"batch_size":1}},
 "6": {"class_type":"LTXVConditioning","inputs":{"positive":["3",0],"negative":["4",0],"frame_rate":24.0}},
 "7": {"class_type":"KSampler","inputs":{"seed":$SEED,"steps":30,"cfg":3.0,"sampler_name":"euler","scheduler":"normal","denoise":1.0,"model":["1",0],"positive":["6",0],"negative":["6",1],"latent_image":["5",0]}},
 "8": {"class_type":"VAEDecode","inputs":{"samples":["7",0],"vae":["1",2]}},
 "9": {"class_type":"VHS_VideoCombine","inputs":{"images":["8",0],"frame_rate":24,"format":"video/h264-mp4","filename_prefix":"gen","pingpong":false,"save_output":true}}
}''',
"voice": '''{
 "1": {"class_type":"KokoroSpeaker","inputs":{"speaker_name":"$VOICE"}},
 "2": {"class_type":"KokoroGenerator","inputs":{"text":"$PROMPT","speaker":["1",0],"speed":1.0,"lang":"English"}},
 "3": {"class_type":"SaveAudio","inputs":{"audio":["2",0],"filename_prefix":"gen"}}
}'''
}

def run(modality, prompt, params=None):
    params = params or {}
    raw = TEMPLATES[modality]
    seed = params.get('seed') or int.from_bytes(os.urandom(4),'big')
    repl = {'$PROMPT':prompt.replace('"',"'").replace('\n',' '),'$SEED':str(seed),
            '$WIDTH':str(params.get('width',1024)),'$HEIGHT':str(params.get('height',1024)),
            '$FRAMES':str(int(params.get('seconds',5)*24)+1),'$VOICE':params.get('voice','af_heart')}
    for k,v in repl.items(): raw = raw.replace(k,v)
    wf = json.loads(raw)  # text-replace BEFORE parse
    cid = str(uuid.uuid4())
    req = urllib.request.Request(f'http://{HOST}/prompt',
            data=json.dumps({'prompt':wf,'client_id':cid}).encode(),
            headers={'Content-Type':'application/json'})
    pid = json.loads(urllib.request.urlopen(req).read())['prompt_id']
    ws = websocket.WebSocket(); ws.connect(f'ws://{HOST}/ws?clientId={cid}')
    while True:
        m = ws.recv()
        if not isinstance(m,str): continue
        d = json.loads(m)
        if d.get('type')=='executing' and d['data'].get('node') is None and d['data'].get('prompt_id')==pid: break
    ws.close()
    h = json.loads(urllib.request.urlopen(f'http://{HOST}/history/{pid}').read())[pid]
    for node in h['outputs'].values():
        for key in ('images','gifs','audio','videos'):
            if key in node:
                f = node[key][0]
                q = urllib.parse.urlencode({'filename':f['filename'],'subfolder':f.get('subfolder',''),'type':f.get('type','output')})
                data = urllib.request.urlopen(f'http://{HOST}/view?{q}').read()
                p = f"/content/out_{modality}.{f['filename'].split('.')[-1]}"
                open(p,'wb').write(data); return p
    raise RuntimeError('no output')
print('run() ready')

In [ ]:
# 5a. IMAGE — cheapest, test first
from IPython.display import Image, Video, Audio, display
p = run('image','a red fox in fresh snow, cinematic lighting, sharp focus', {'width':1024,'height':1024,'seed':42})
display(Image(p))

In [ ]:
# 5b. VOICE
p = run('voice','Welcome to GenStudio, where your ideas become video, image, and speech.', {'voice':'af_heart'})
display(Audio(p))

In [ ]:
# 5c. VIDEO — slowest, ~1-2 min on T4
p = run('video','a paper airplane gliding over a city skyline at sunset', {'width':768,'height':512,'seconds':5,'seed':7})
display(Video(p, embed=True))

## 6. If a node errors: re-export the fixed workflow
Usually a custom-node class-name mismatch (Kokoro `KokoroGenerate`, or `VHS_VideoCombine`).
Open the ComfyUI UI, rebuild that graph, **Settings → enable dev mode → Save (API Format)**,
then paste the node dict into the matching `TEMPLATES` entry above (keep the `$TOKEN`s).

Reach the UI from Colab:
```python
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(8188)
```